In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score
import seaborn as sns
# 加载数据集
df = pd.read_csv('tips.csv')
print("原始数据形状:", df.shape)
df.head()

原始数据形状: (244, 7)


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [4]:
cols = ['sex', 'smoker', 'day']

# 不删除第一类
ohe_no_drop = OneHotEncoder(drop=None, sparse_output=False)
encoded_no_drop = ohe_no_drop.fit_transform(df[cols])
print("不删除第一类，特征列数:", encoded_no_drop.shape[1])

# 删除第一类
ohe_drop = OneHotEncoder(drop='first', sparse_output=False)
encoded_drop = ohe_drop.fit_transform(df[cols])
print("删除第一类后，特征列数:", encoded_drop.shape[1])

# 单独处理 time 列
time_ohe_no_drop = OneHotEncoder(drop=None, sparse_output=False)
time_enc = time_ohe_no_drop.fit_transform(df[['time']])
print("time 列独热编码（不删除）列数:", time_enc.shape[1])

time_ohe_drop = OneHotEncoder(drop='first', sparse_output=False)
time_enc_drop = time_ohe_drop.fit_transform(df[['time']])
print("time 列独热编码（删除第一类）列数:", time_enc_drop.shape[1])

不删除第一类，特征列数: 8
删除第一类后，特征列数: 5
time 列独热编码（不删除）列数: 2
time 列独热编码（删除第一类）列数: 1


In [5]:
X = df[['size']].copy()
y = df['tip']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 序数编码（明确顺序 1-6）
ord_enc = OrdinalEncoder(categories=[[1,2,3,4,5,6]], dtype=int)
X_train_ord = ord_enc.fit_transform(X_train)
X_test_ord = ord_enc.transform(X_test)

# 独热编码（删除第一类避免共线性）
ohe = OneHotEncoder(drop='first', sparse_output=False)
X_train_ohe = ohe.fit_transform(X_train)
X_test_ohe = ohe.transform(X_test)

# 线性回归
lr_ord = LinearRegression().fit(X_train_ord, y_train)
lr_ohe = LinearRegression().fit(X_train_ohe, y_train)

r2_ord = r2_score(y_test, lr_ord.predict(X_test_ord))
r2_ohe = r2_score(y_test, lr_ohe.predict(X_test_ohe))

print(f"序数编码 R²: {r2_ord:.4f}")
print(f"独热编码 R²: {r2_ohe:.4f}")

序数编码 R²: 0.1049
独热编码 R²: 0.0305


In [6]:
def target_encode_cv_debug(
    train: pd.DataFrame,
    col: str,
    target: str,
    n_splits: int = 3,
):
    global_mean = train[target].mean()
    encoded = pd.Series(np.nan, index=train.index)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
        fold_train = train.iloc[train_idx]
        fold_val   = train.iloc[val_idx]
        print(f"Fold {fold}: train size = {len(train_idx)}, val size = {len(val_idx)}")
        
        # 仅使用 fold_train 计算均值，不包含任何 fold_val 的目标值
        stats = fold_train.groupby(col)[target].mean()
        encoded.iloc[val_idx] = fold_val[col].map(stats).fillna(global_mean)
    
    return encoded

# 测试
df_test = df[['size', 'tip']].copy()
df_test['size_encoded'] = target_encode_cv_debug(df_test, 'size', 'tip', n_splits=3)
print("\n编码结果示例:\n", df_test.head())

Fold 0: train size = 162, val size = 82
Fold 1: train size = 163, val size = 81
Fold 2: train size = 163, val size = 81

编码结果示例:
    size   tip  size_encoded
0     2  1.01      2.606095
1     3  1.66      3.510000
2     3  3.50      3.103750
3     2  3.31      2.562929
4     4  3.61      3.842917


In [7]:
# 训练集只有三种颜色
train = pd.DataFrame({'color': ['red', 'blue', 'green']})
# 测试集包含新颜色 'yellow'
test = pd.DataFrame({'color': ['red', 'yellow', 'blue']})

ohe_ignore = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe_ignore.fit(train[['color']])

train_enc = ohe_ignore.transform(train[['color']])
test_enc = ohe_ignore.transform(test[['color']])

print("训练集编码:\n", train_enc)
print("测试集编码:\n", test_enc)
print("特征名称:", ohe_ignore.get_feature_names_out())

训练集编码:
 [[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]
测试集编码:
 [[0. 0. 1.]
 [0. 0. 0.]
 [1. 0. 0.]]
特征名称: ['color_blue' 'color_green' 'color_red']
